# 股関節DDR｜LCEA・FEAR index 自動計測（新規学習・左右局所Pose補正版）

**学会用探索的解析 / 2026年10月 / Google Colab GPU推奨**

## この改訂版の考え方

以前のPA 14点一括モデルでは、右 `lateral_sourcil_R` が反対側へ約500 px移動する誤検出が発生しました。今回は新しい2つのZIPのみを使用し、以下の**二段階Pose**へ変更します。

1. **全骨盤10点Pose**：teardrop左右、femoral head左右等を検出。主に骨盤水平軸と左右股関節中心を取得。
2. **局所股関節5点Pose**：各股関節を中心に480px四方で切り出し、**右股関節を水平反転**して左右とも外側が画面右方向になるよう統一。`femoral head / lateral sourcil / medial sourcil / physeal scar medial / physeal scar lateral`を予測。左右両股を共通モデルで学習。
3. **二段階の推論**：全骨盤Pose → 推定骨頭中心から左右ROI作成 → 局所Pose → 画像座標に逆変換 → LCEA / FEAR-like傾斜差を計算。
4. **厳格なQC**：左右の解剖学的位置関係、点間距離、予測信頼度を監査。不自然な値は明示的に `NaN` とし、勝手に補正しない。

**モデル初期化**：`yolo11n-pose.pt`（一般的COCO事前学習重み）を初期値に使用しますが、**過去の股関節モデルは一切使いません**。完全ランダム初期化したい場合は `USE_GENERIC_PRETRAINED=False` へ変更できます。ただし、この少数データでは一般的事前学習の利用を推奨します。

**対象**：今回のZIPに入っている健常PA 9患者174フレーム、両側DDH PA 3患者60フレーム、MLは健常10患者192フレーム＋DDH3患者62フレーム。新ZIPは旧5例のPAを含むため、旧 `Annotation PA.zip` を追加しません。

**臨床上の注意**：LCEAはteardrop基準の骨盤垂直線を用いた2D動的投影角です。FEARについては原著の符号規約との確認前に「FEAR-like角度差／探索的FEAR指標」と記載してください。推論が正常に完了することと臨床的に妥当な値が得られることは別です。

**統計**：13患者（健常10名、両側DDH3名）を独立単位とし、フレーム数や左右股数を独立症例数として検定しません。LCEA/FEARの手動参照があるPAは健常4名・DDH3名です。有意差検定、診断AUCは実施しません。

## 0. ランタイムとファイルの準備
Colab GPU（T4等）を選択。2つのZIPを `/content/` にアップロードするか、下のパスをGoogle Driveパスへ変更してください。過去の `.pt` / 旧ZIPは不要です。

In [ ]:
%pip -q install ultralytics opencv-python-headless pandas numpy matplotlib pyyaml

In [ ]:
import re, os, json, io, math, zipfile, shutil, time, gc, random
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np, pandas as pd, matplotlib.pyplot as plt, yaml, cv2
import torch
from ultralytics import YOLO
from IPython.display import display, HTML
from base64 import b64encode

# -------- USER CONFIG ----------
HEALTHY_ZIP = Path('/content/AI-Hip Gate Analysis.zip')
DDH_ZIP = Path('/content/gigafile-1022-d973a9d3dde722592c63b5704b11c478.zip')
ROOT = Path('/content/DDR_DDH_Revised_2026')

# 学会直前の推奨値: global 100 epoch + local 130 epoch / no past model
USE_GENERIC_PRETRAINED = True   # COCO一般事前学習。旧股関節.ptは使わない
FORCE_RETRAIN = False
RUN_TRAINING = True
GLOBAL_EPOCHS = 120
LOCAL_EPOCHS = 150
GLOBAL_IMGSZ = 960
LOCAL_IMGSZ = 640
GLOBAL_BATCH = 4
LOCAL_BATCH = 8
CROP_SIZE = 480
LOCAL_JITTER_PX = (0, 32, 55)  # 3種類/股関節/フレーム。trainingだけjitter
PRED_CONF = 0.08
KP_CONF = 0.15
SEED = 42

# 患者単位: 学習8例 / validation2例 / test2例
TEST_PATIENTS = ('010', 'DDH003')
VAL_PATIENTS = ('009', 'DDH002')

# 時間があるときのみ（各foldでglobal+localの2モデルを学習）
RUN_LOSO = False
LOSO_CASES = ('006','008','009','010','DDH001','DDH002','DDH003')
LOSO_MAX_FOLDS = None   # 試走は1 or 2。最終評価はNone
LOSO_GLOBAL_EPOCHS = 80
LOSO_LOCAL_EPOCHS = 100

# 側面ML：初回はPA最優先。後日対応可能な補助解析として任意実行
TRAIN_ML = False
ML_EPOCHS = 100

random.seed(SEED)
np.random.seed(SEED)
ROOT.mkdir(parents=True, exist_ok=True)
for p in (HEALTHY_ZIP,DDH_ZIP):
    if not p.exists():
        raise FileNotFoundError(f'見つかりません：{p}。Colab左欄にZIPをアップロードしてください。')
print('GPU:',torch.cuda.is_available(), 'Workspace:',ROOT)

## 1. COCO ZIPを解凍せずに読み込み、category名とkeypoint名から統合
左右の点の**カテゴリ配列順**は異なります（例：`physeal scar R` は `Lateral, medial`）。`category_id`や点の配列番号を横断流用せず、**名前**で正規化します。`LCEA R`と`LCEA L`の別注釈は独立手動参照として保持します。

In [ ]:
PA_BASE = ['lateral_sourcil_R','lateral_sourcil_L','medial_sourcil_R','medial_sourcil_L',
           'teardrop_R','teardrop_L','femoral_head_R','femoral_head_L','sacrum_R','sacrum_L']
LOCAL_KP = ['femoral_head','lateral_sourcil','medial_sourcil','scar_medial','scar_lateral']
ML_KP = ['PT_L_FL','PT_L_S1','PT_R_FR','PT_R_S1','S1_A','S1_C','S1_V','Sacrum_Proximal','Sacrum_Distal']
CAT_MAP = {'lateral sourci':'lateral_sourcil','medial sourci':'medial_sourcil',
           'tear drop':'teardrop','femoral head':'femoral_head','sacrum':'sacrum',
           'physeal scar r':'scar_R','physeal scar l':'scar_L',
           'lcea l':'LCEA_L','lcea r':'LCEA_R','pt l':'PT_L','pt r':'PT_R','s1':'S1'}

def normalize_name(s):
    return re.sub(r'\s+',' ',str(s).replace('_',' ').replace('-',' ').strip().lower())

def zip_patient_view(filename):
    m = re.search(r'--(DDH\d+|\d+)\s+(PA|ML)',filename,re.I)
    if not m: raise ValueError('inner zip name unexpected: '+filename)
    subj=m.group(1).upper()
    if subj.isdigit(): subj=subj.zfill(3)
    return subj,m.group(2).upper()

def get_frame_number(filename):
    m=re.search(r'(?:mp4|avi)[-_](\d+)',filename,re.I)
    if not m:m=re.search(r'-(\d+)_jpg',filename,re.I)
    if not m:raise ValueError('frame番号を解釈できません: '+filename)
    return int(m.group(1))

def parse_annotation(ann,cat):
    d={}; array=ann.get('keypoints',[])
    for i,k in enumerate(cat.get('keypoints',[])):
        if 3*i+2>=len(array):continue
        x,y,v=array[i*3:i*3+3]
        if v>0 and np.isfinite(x) and np.isfinite(y):
            d[normalize_name(k)]=(float(x),float(y),int(v))
    return d

def extract_landmarks(annotations,cat_map):
    by=defaultdict(list)
    for a in annotations:
        cat=cat_map.get(a['category_id'])
        if cat is None:continue
        n=CAT_MAP.get(normalize_name(cat['name']))
        if n:by[n].append((a,cat))
    points={}; lcea_ref={}; dup={k:len(v) for k,v in by.items() if len(v)>1}
    for cname,candidates in by.items():
        # 多数派を選ぶのではなく、可視点の多いannotationを使用
        a,cat=max(candidates,key=lambda t:sum(v>0 for v in t[0].get('keypoints',[])[2::3]))
        p=parse_annotation(a,cat)
        if cname in ('lateral_sourcil','medial_sourcil','teardrop','femoral_head'):
            for side in ('R','L'):
                if side.lower() in p:points[f'{cname}_{side}']=p[side.lower()]
        elif cname=='sacrum':
            for side in ('R','L'):
                if side.lower() in p:points[f'sacrum_{side}']=p[side.lower()]
            for k in ('proximal','distal'):
                if k in p:points[f'Sacrum_{k.title()}']=p[k]
        elif cname in ('scar_R','scar_L'):
            s=cname[-1]
            for k in ('medial','lateral'):
                if k in p:points[f'scar_{s}_{k}']=p[k]
        elif cname in ('LCEA_R','LCEA_L'):
            s=cname[-1]
            if 'center' in p:lcea_ref[f'LCEA_{s}_center']=p['center']
            edge=p.get('sourcil l',p.get('sourcil'))
            if edge is not None:lcea_ref[f'LCEA_{s}_edge']=edge
        elif cname in ('PT_L','PT_R'):
            s=cname[-1]; other='fl' if s=='L' else 'fr'
            if other in p:points[f'PT_{s}_{other.upper()}']=p[other]
            if 's1' in p:points[f'PT_{s}_S1']=p['s1']
        elif cname=='S1':
            for k in ('a','c','v'):
                if k in p:points[f'S1_{k.upper()}']=p[k]
    return points,lcea_ref,dup

RAW_DIR=ROOT/'raw'; RAW_DIR.mkdir(parents=True,exist_ok=True)
records=[]; dup_rows=[]; identities=set()
for origin,zippath in [('Healthy',HEALTHY_ZIP),('DDH',DDH_ZIP)]:
    with zipfile.ZipFile(zippath) as outer:
        for inner_name in sorted(n for n in outer.namelist() if n.lower().endswith('.zip')):
            patient,view=zip_patient_view(inner_name)
            group='DDH' if patient.startswith('DDH') else 'Healthy'
            if group!=origin:raise ValueError(f'ZIPと群の整合性がありません: {inner_name}')
            with zipfile.ZipFile(io.BytesIO(outer.read(inner_name))) as inner:
                jnames=[n for n in inner.namelist() if n.lower().endswith('.json')]
                if len(jnames)!=1:raise ValueError((inner_name,jnames))
                coco=json.loads(inner.read(jnames[0]))
                categories={c['id']:c for c in coco['categories']}
                ann_map=defaultdict(list)
                for a in coco['annotations']:ann_map[a['image_id']].append(a)
                img_lookup={Path(n).name:n for n in inner.namelist() if n.lower().endswith(('.jpg','.png','.jpeg'))}
                for im in coco['images']:
                    frame=get_frame_number(im['file_name'])
                    ident=(patient,view,frame)
                    if ident in identities:raise ValueError('重複動画frame:'+str(ident))
                    identities.add(ident)
                    pts,refs,dups=extract_landmarks(ann_map.get(im['id'],[]),categories)
                    if dups:dup_rows.append(dict(patient=patient,view=view,frame=frame,duplicates=str(dups)))
                    internal=img_lookup.get(Path(im['file_name']).name)
                    if not internal:raise FileNotFoundError(im['file_name'])
                    ext=Path(internal).suffix.lower()
                    filename=f'{view}_{patient}_f{frame:04d}{ext}'
                    target=RAW_DIR/view/filename;target.parent.mkdir(parents=True,exist_ok=True)
                    with inner.open(internal) as inp,open(target,'wb') as out:shutil.copyfileobj(inp,out)
                    records.append({'patient':patient,'group':group,'view':view,'frame':frame,
                                    'filename':filename,'path':str(target),'width':im['width'],
                                    'height':im['height'],'points':pts,'ref':refs})
manifest=pd.DataFrame([{k:v for k,v in r.items() if k not in ('points','ref')} for r in records])
manifest.to_csv(ROOT/'dataset_manifest.csv',index=False)
pd.DataFrame(dup_rows,columns=['patient','view','frame','duplicates']).to_csv(ROOT/'annotation_duplicates_QC.csv',index=False)
print(manifest.groupby(['group','view']).agg(patients=('patient','nunique'),frames=('frame','count')))
print('全患者',manifest.patient.nunique(),'全画像',len(manifest))
assert len(records)==488, f'想定488枚、実際は{len(records)}枚。ZIPの版が違う可能性があります。'

## 2. アノテーションの生体幾何QC
異常な点は**元データを勝手に移動しません**。`annotation_geometry_QC.csv`に座標・左右関係の検査結果を残し、学習時には明らかに矛盾する点だけ無効化（`v=0`）できるようにします。これは**機械的な位置QC**であり、骨性ランドマークの医学的な妥当性を保証しません。

In [ ]:
# 明瞭な位置異常のみ：Rは画面左、Lは画面右、左右teardrop中心で判定
# 診断群の角度差から何かを補正したり、正解座標を作り直したりはしない。
qc=[]
for r in records:
    if r['view']!='PA':continue
    p=r['points']
    tr=p.get('teardrop_R');tl=p.get('teardrop_L')
    mid=(tr[0]+tl[0])/2 if tr and tl else r['width']/2
    for side in ('R','L'):
        h=p.get(f'femoral_head_{side}')
        outer=p.get(f'lateral_sourcil_{side}')
        medial=p.get(f'medial_sourcil_{side}')
        reason=[]
        if h and ((side=='R' and h[0]>=mid) or (side=='L' and h[0]<=mid)):reason.append('head_wrong_half')
        if outer and h:
            # 外側はRで小さいx、Lで大きいx（画像上の定義）
            dx=(outer[0]-h[0]) * (1 if side=='L' else -1)
            if dx < -30:reason.append('edge_inward_over30px')
            if abs(outer[0]-h[0])>260:reason.append('edge_far_from_head')
        if medial and outer:
            d=(outer[0]-medial[0])*(1 if side=='L' else -1)
            if d < -15:reason.append('sourcil_anatomical_order')
        qc.append({'patient':r['patient'],'group':r['group'],'frame':r['frame'],'side':side,
                   'head_x':h[0] if h else np.nan,'lateral_x':outer[0] if outer else np.nan,
                   'medial_x':medial[0] if medial else np.nan,'midline_x':mid,'qc_flags':'|'.join(reason)})
qcd=pd.DataFrame(qc);qcd.to_csv(ROOT/'annotation_geometry_QC.csv',index=False)
print('異常フレーム（アノテーション段階）:',int(qcd.qc_flags.ne('').sum()))
display(qcd[qcd.qc_flags.ne('')].head(20))

# 欠損内訳：旧PA 001-005にはscarの正解annotationがないため0,0,0を使用
presence=[]
for r in records:
    names=PA_BASE+[f'scar_{s}_{m}' for s in ('R','L') for m in ('medial','lateral')] if r['view']=='PA' else ML_KP
    for n in names:
        presence.append({'patient':r['patient'],'group':r['group'],'view':r['view'],
                         'frame':r['frame'],'landmark':n,'present':int(n in r['points'])})
presence_df=pd.DataFrame(presence);presence_df.to_csv(ROOT/'annotation_presence.csv',index=False)
display(presence_df[(presence_df.view=='PA') & presence_df.landmark.str.startswith('scar')].groupby(['patient','landmark']).present.agg(['sum','count']))

## 3. 患者分割・train/val/testの自動作成
全モデルで同じ患者分割を使います。同じ人のPA/MLや連続フレームがtestとtrainにまたがらないようにします。

In [ ]:
all_patients=set(manifest.patient.unique())
assert set(TEST_PATIENTS).issubset(all_patients) and set(VAL_PATIENTS).issubset(all_patients)
assert not (set(TEST_PATIENTS)&set(VAL_PATIENTS))

def which_split(patient,test=TEST_PATIENTS,val=VAL_PATIENTS):
    return 'test' if patient in test else ('val' if patient in val else 'train')

split_df=manifest[['patient','group','view','frame','filename']].copy()
split_df['split']=split_df.patient.map(which_split)
split_df.to_csv(ROOT/'patient_split_manifest.csv',index=False)
display(split_df.groupby(['view','group','split']).agg(n_patients=('patient','nunique'),n_frames=('frame','size')))
for v in ('PA','ML'):
    for pat,g in split_df[split_df.view==v].groupby('patient'):
        assert g.split.nunique()==1, '患者漏洩!'
print('Patient-level splitting: OK')

## 4. 全骨盤10点ラベル作成
`lateral_sourcil_R`は全骨盤モデルにも残しますが、最終角度に使用する外側縁は**局所モデルの再推定**を優先します。全骨盤モデルは骨頭中心と左右teardropによる局所化と基準軸が主目的です。

In [ ]:
DATA=ROOT/'yolo_data'

def safe_label_bbox(points,w,h,pad=.065):
    pts=np.array(points,dtype=float)
    if len(pts)<2:return None
    lo=np.clip(pts.min(axis=0)/[w,h]-pad,0,1)
    hi=np.clip(pts.max(axis=0)/[w,h]+pad,0,1)
    b=[(lo[0]+hi[0])/2,(lo[1]+hi[1])/2,hi[0]-lo[0],hi[1]-lo[1]]
    return b

def write_pose_label(path,w,h,names,points):
    kpts=[];used=[]
    for n in names:
        p=points.get(n)
        if p is None or p[2]<=0 or not (0<=p[0]<w and 0<=p[1]<h):
            kpts.extend([0.,0.,0]);continue
        x,y=float(p[0]),float(p[1]);used.append((x,y))
        kpts.extend([x/w,y/h,2])
    bb=safe_label_bbox(used,w,h)
    if bb is None: path.write_text('');return False
    arr=[0]+bb+kpts
    path.write_text(' '.join(f'{x:.7f}' if isinstance(x,float) else str(x) for x in arr)+'\n')
    return True

def write_yaml(folder,names,viewname):
    obj={'path':str(folder),'train':'images/train','val':'images/val','test':'images/test',
         'nc':1,'names':[viewname],'kpt_shape':[len(names),3],'flip_idx':list(range(len(names)))}
    yml=folder/'data.yaml';folder.mkdir(parents=True,exist_ok=True)
    with open(yml,'w') as f:yaml.safe_dump(obj,f,sort_keys=False)
    return yml

def prepare_global(view='PA',test=TEST_PATIENTS,val=VAL_PATIENTS,root=None):
    dataset=(Path(root) if root else DATA/f'global_{view}')
    if dataset.exists():shutil.rmtree(dataset)
    names=PA_BASE if view=='PA' else ML_KP
    label_count=0
    for r in records:
        if r['view']!=view:continue
        split=which_split(r['patient'],test,val)
        p=dataset/'images'/split/r['filename'];p.parent.mkdir(parents=True,exist_ok=True)
        shutil.copy2(r['path'],p)
        label=dataset/'labels'/split/Path(r['filename']).with_suffix('.txt').name
        label.parent.mkdir(parents=True,exist_ok=True)
        label_count+=write_pose_label(label,r['width'],r['height'],names,r['points'])
    return dataset,write_yaml(dataset,names,'pelvis_'+view),label_count

GLOBAL_DATA,GLOBAL_YAML,global_valid=prepare_global('PA')
print('Global valid labels:',global_valid,'YAML:',GLOBAL_YAML)

## 5. 左右股関節を共通方向にそろえる局所Poseデータ
左はそのまま、**右は左右反転して外側が右向き**になるようにします。trainにはGT骨頭中心でROIを作り、位置ずれに耐えるため中心を±32/55px揺らした追加ROIも生成します。val/testにはGT中心のROIを作りますが、**最終評価ではこれを使わず必ずAIの全骨盤予測でROIを作ります**（GT中心の混入による楽観評価を防ぐため）。

In [ ]:
def crop_square_with_padding(image,cx,cy,size):
    """任意実数中心の正方形crop。外側は黒埋め。offset(x0,y0)を返す。"""
    x0=int(round(cx-size/2));y0=int(round(cy-size/2));x1=x0+size;y1=y0+size
    H,W=image.shape[:2]
    patch=np.zeros((size,size,3),dtype=np.uint8)
    sx0=max(0,x0);sy0=max(0,y0);sx1=min(W,x1);sy1=min(H,y1)
    if sx1>sx0 and sy1>sy0:
        patch[sy0-y0:sy1-y0,sx0-x0:sx1-x0]=image[sy0:sy1,sx0:sx1]
    return patch,x0,y0

def to_local_point(p,x0,y0,size,side):
    if p is None or p[2]<=0:return None
    x=p[0]-x0;y=p[1]-y0
    if side=='R':x=(size-1)-x  # right -> normalized lateral-right
    if not (0<=x<size and 0<=y<size):return None
    return (float(x),float(y),2)

def from_local_point(p,x0,y0,size,side):
    if p is None:return None
    x=float(p[0]);y=float(p[1])
    if side=='R':x=(size-1)-x
    return (x+x0,y+y0,float(p[2]) if len(p)>2 else 2)

def make_local_points(r,side,x0,y0,size):
    mp={}
    canonical={'femoral_head':f'femoral_head_{side}',
               'lateral_sourcil':f'lateral_sourcil_{side}',
               'medial_sourcil':f'medial_sourcil_{side}',
               'scar_medial':f'scar_{side}_medial',
               'scar_lateral':f'scar_{side}_lateral'}
    for name,src in canonical.items():
        p=to_local_point(r['points'].get(src),x0,y0,size,side)
        if p is not None:mp[name]=p
    return mp

def prepare_local(test=TEST_PATIENTS,val=VAL_PATIENTS,root=None,aug=True):
    dataset=Path(root) if root else DATA/'local_hip'
    if dataset.exists():shutil.rmtree(dataset)
    rng=np.random.default_rng(SEED)
    local_rows=[];nvalid=0
    for r in records:
        if r['view']!='PA':continue
        sp=which_split(r['patient'],test,val)
        image=cv2.imread(r['path'])
        for side in ('R','L'):
            head=r['points'].get(f'femoral_head_{side}')
            if head is None:continue
            js=LOCAL_JITTER_PX if sp=='train' and aug else (0,)
            for augidx,j in enumerate(js):
                dx=dy=0 if j==0 else None
                if j:
                    dx=int(rng.integers(-j,j+1));dy=int(rng.integers(-j,j+1))
                patch,x0,y0=crop_square_with_padding(image,head[0]+dx,head[1]+dy,CROP_SIZE)
                if side=='R':patch=cv2.flip(patch,1)
                pts=make_local_points(r,side,x0,y0,CROP_SIZE)
                filename=f'{Path(r["filename"]).stem}_{side}_a{augidx}.jpg'
                path=dataset/'images'/sp/filename;path.parent.mkdir(parents=True,exist_ok=True)
                cv2.imwrite(str(path),patch,[cv2.IMWRITE_JPEG_QUALITY,95])
                lab=dataset/'labels'/sp/Path(filename).with_suffix('.txt').name
                lab.parent.mkdir(parents=True,exist_ok=True)
                nvalid+=write_pose_label(lab,CROP_SIZE,CROP_SIZE,LOCAL_KP,pts)
                local_rows.append({'patient':r['patient'],'group':r['group'],'frame':r['frame'],
                    'image':filename,'split':sp,'side':side,'center_x':head[0]+dx,'center_y':head[1]+dy,
                    'crop_x0':x0,'crop_y0':y0,'scar_present':int('scar_medial' in pts and 'scar_lateral' in pts)})
    df=pd.DataFrame(local_rows)
    df.to_csv(dataset/'local_crop_manifest.csv',index=False)
    return dataset,write_yaml(dataset,LOCAL_KP,'single_hip'),df,nvalid

LOCAL_DATA,LOCAL_YAML,LOCAL_MANIFEST,local_valid=prepare_local()
print('Local valid labels:',local_valid)
display(LOCAL_MANIFEST.groupby(['split','group']).agg(hips=('image','size'),subjects=('patient','nunique')))

# ラベルの方向・部位を3画像で確認
for sp in ('train','test'):
    samples=LOCAL_MANIFEST[LOCAL_MANIFEST.split==sp].sample(n=min(2,(LOCAL_MANIFEST.split==sp).sum()),random_state=SEED)
    fig,ax=plt.subplots(1,len(samples),figsize=(6*len(samples),6),squeeze=False)
    for a,(_,s) in zip(ax[0],samples.iterrows()):
        im=cv2.cvtColor(cv2.imread(str(LOCAL_DATA/'images'/sp/s.image)),cv2.COLOR_BGR2RGB)
        a.imshow(im)
        label=(LOCAL_DATA/'labels'/sp/Path(s.image).with_suffix('.txt').name).read_text().split()
        if label:
            kps=np.array(list(map(float,label[5:]))).reshape(-1,3)
            for i,p in enumerate(kps):
                if p[2]>0:a.scatter(p[0]*CROP_SIZE,p[1]*CROP_SIZE);a.text(p[0]*CROP_SIZE+3,p[1]*CROP_SIZE,str(i),fontsize=10)
        a.set_title(s.image);a.axis('off')
    plt.tight_layout();plt.show()

## 6. 手動アノテーションからLCEA／FEAR-likeを計算してQC
既存の座標から角度を算出します。**この節はAIの精度を示すものではありません。** LCEA独立アノテーションとの比較は、計測定義差も含むため別列に保存します。

In [ ]:
def getxy(pts,key):
    p=pts.get(key)
    if p is None or len(p)<2 or not np.isfinite(p[0]) or not np.isfinite(p[1]):return None
    if len(p)>=3 and p[2]<=0:return None
    return np.array(p[:2],dtype=float)

def pelvic_frame(pts):
    tr=getxy(pts,'teardrop_R');tl=getxy(pts,'teardrop_L')
    if tr is None or tl is None:return None
    h=tl-tr
    norm=np.linalg.norm(h)
    if norm<80 or h[0]<40:return None
    h=h/norm
    up=np.array([h[1],-h[0]])
    return h,up

def signed_lcea(pts,side,use_ref=False):
    h=getxy(pts,f'LCEA_{side}_center' if use_ref else f'femoral_head_{side}')
    edge=getxy(pts,f'LCEA_{side}_edge' if use_ref else f'lateral_sourcil_{side}')
    ax=pelvic_frame(pts)
    if ax is None or h is None or edge is None:return np.nan
    horiz,up=ax;out=1 if side=='L' else -1
    v=edge-h
    # ランドマーク位置を確認せずに鈍角を作らない
    dx=float(np.dot(v,out*horiz));dy=float(np.dot(v,up))
    if not (8<np.linalg.norm(v)<240 and 0<dx<220 and dy>5):return np.nan
    return float(np.degrees(np.arctan2(dx,dy)))

def tilt_from_medial_to_lateral(pts,side,prefix):
    ax=pelvic_frame(pts)
    if ax is None:return np.nan
    horiz,up=ax
    a=getxy(pts,f'{prefix}_{side}_medial' if prefix=='scar' else f'medial_sourcil_{side}')
    b=getxy(pts,f'{prefix}_{side}_lateral' if prefix=='scar' else f'lateral_sourcil_{side}')
    if a is None or b is None:return np.nan
    v=b-a;out=(1 if side=='L' else -1)*horiz
    dx=float(v@out);down=float(v@(-up))
    if not (8<dx<230):return np.nan
    return float(np.degrees(np.arctan2(down,dx)))

def fear_like(pts,side):
    # ここではscar線とroof線の相対傾斜。原著FEARとの符号規約は臨床者による要検証。
    roof=tilt_from_medial_to_lateral(pts,side,'sourcil')
    scar=tilt_from_medial_to_lateral(pts,side,'scar')
    if not (np.isfinite(roof) and np.isfinite(scar)):return np.nan
    d=(scar-roof+90)%180-90  # 有向でなく直線差（-90〜90度）
    return float(d)

def manual_angle_table(records):
    rows=[]
    for r in records:
        if r['view']!='PA':continue
        for side in ('R','L'):
            pts=r['points'];ind={**pts,**r['ref']}
            rows.append({'patient':r['patient'],'group':r['group'],'frame':r['frame'],'side':side,
                'filename':r['filename'],'lcea_from_landmarks_deg':signed_lcea(pts,side),
                'lcea_independent_annotation_deg':signed_lcea(ind,side,use_ref=True),
                'fear_like_deg':fear_like(pts,side),
                'has_independent_lcea':int(f'LCEA_{side}_center' in r['ref'] and f'LCEA_{side}_edge' in r['ref']),
                'has_scar':int(f'scar_{side}_medial' in pts and f'scar_{side}_lateral' in pts)})
    return pd.DataFrame(rows)
GT=manual_angle_table(records)
GT.to_csv(ROOT/'GT_manual_angle_per_frame.csv',index=False)
GT_SUM=GT.groupby(['patient','group','side'],as_index=False).agg(
    n_frames=('frame','count'),n_lcea=('lcea_independent_annotation_deg','count'),
    LCEA_median=('lcea_independent_annotation_deg','median'),
    n_fear=('fear_like_deg','count'),FEAR_like_median=('fear_like_deg','median'))
GT_SUM.to_csv(ROOT/'GT_patient_hip_exploratory.csv',index=False)
display(GT_SUM)
qc_fear=GT.loc[GT.fear_like_deg.abs()>35].copy()
qc_fear.to_csv(ROOT/'FEAR_manual_outlier_QC.csv',index=False)
print('FEAR-like angle abs>35° 要レビュー:',len(qc_fear))

## 7. 全骨盤10点Pose学習（新データのみ）
`USE_GENERIC_PRETRAINED=True`でも、以前の股関節研究の重みは一切読み込みません。COCO一般重みを用い、keypoint headを今回の10点に再構成して学習します。GPUメモリ不足なら `GLOBAL_BATCH=2`, `GLOBAL_IMGSZ=768`へ。

In [ ]:
def cleanup_gpu(tag=''):
    plt.close('all');gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(tag,'GPU alloc GB:',round(torch.cuda.memory_allocated()/1024**3,2),
                    'cached GB:',round(torch.cuda.memory_reserved()/1024**3,2))

def train_pose(yaml_path,run_dir,epochs,imgsz,batch,run_enabled=True,force=False,seed=SEED):
    weight=run_dir/'weights'/'best.pt'
    if weight.exists() and not force:
        print('既存の学習済みcheckpoint使用:',weight)
        return weight
    if not run_enabled:raise RuntimeError('RUN_TRAINING=Falseだがbest.ptが存在しません: '+str(weight))
    cleanup_gpu('before train')
    # UltralyticsのCOCO一般学習済みモデル。股関節旧重みは一切ロードしない。
    model=YOLO('yolo11n-pose.pt' if USE_GENERIC_PRETRAINED else 'yolo11n-pose.yaml')
    result=model.train(data=str(yaml_path),epochs=epochs,imgsz=imgsz,batch=batch,patience=40,
        project=str(run_dir.parent),name=run_dir.name,device=0 if torch.cuda.is_available() else 'cpu',
        seed=seed,deterministic=True,cache=False,workers=2,amp=torch.cuda.is_available(),
        plots=False,exist_ok=True,save_period=-1,
        degrees=3,translate=.06,scale=.08,fliplr=0.0,flipud=0.0,mosaic=0.0,
        hsv_h=0.0,hsv_s=0.0,hsv_v=0.12,
        perspective=0.0,shear=0.0,close_mosaic=0)
    del result,model
    cleanup_gpu('after train')
    assert weight.exists(),weight
    return weight

GLOBAL_RUN=ROOT/'runs'/'global_PA'
GLOBAL_PT=train_pose(GLOBAL_YAML,GLOBAL_RUN,GLOBAL_EPOCHS,GLOBAL_IMGSZ,GLOBAL_BATCH,
                     RUN_TRAINING,FORCE_RETRAIN)
print('Global model:',GLOBAL_PT)

## 8. 左右統一・局所股関節5点Pose学習
5点のうちscar2点は健常001〜005では教師ラベル欠損です。欠損を`0,0,0`としてマスクし、実在しないscarを教師として与えません。

In [ ]:
LOCAL_RUN=ROOT/'runs'/'local_hip_5kp'
LOCAL_PT=train_pose(LOCAL_YAML,LOCAL_RUN,LOCAL_EPOCHS,LOCAL_IMGSZ,LOCAL_BATCH,
                    RUN_TRAINING,FORCE_RETRAIN)
print('Local model:',LOCAL_PT)

## 9. 患者単位hold-outで完全自動（end-to-end）推論
**正解の骨頭中心はtestに使用しません。** Global AIによる骨頭中心からROI作成し、Local AIが角度用点を予測します。局所Pose失敗やQC不合格は**欠測**で残します（穴埋めしません）。

In [ ]:
def best_kps(model,image,imgsz=960,conf=PRED_CONF,n=None):
    res=model.predict(image,imgsz=imgsz,conf=conf,verbose=False,device=0 if torch.cuda.is_available() else 'cpu')[0]
    if res.keypoints is None or len(res.keypoints.xy)==0:return None,{},np.nan
    if res.boxes is not None and len(res.boxes.conf):
        i=int(res.boxes.conf.argmax().item())
        score=float(res.boxes.conf[i].item())
    else:i=0;score=np.nan
    xy=res.keypoints.xy[i].cpu().numpy()
    kpconf=res.keypoints.conf[i].cpu().numpy() if res.keypoints.conf is not None else np.full(len(xy),np.nan)
    pts={}; confidences={}
    for j,name in enumerate(n):
        c=float(kpconf[j])
        if np.isfinite(c) and c<KP_CONF:continue
        if not np.isfinite(xy[j]).all() or np.linalg.norm(xy[j])<1:continue
        pts[name]=(float(xy[j,0]),float(xy[j,1]),2)
        confidences[name]=c
    return pts,confidences,score

def predict_two_stage(global_model,local_model,r):
    im=cv2.imread(r['path'])
    global_pts,gconf,gbox=best_kps(global_model,im,GLOBAL_IMGSZ,n=PA_BASE)
    result={'patient':r['patient'],'group':r['group'],'frame':r['frame'],'filename':r['filename'],
            'global_detected':int(global_pts is not None),'global_box_conf':gbox}
    all_points={} if global_pts is None else dict(global_pts)
    all_confs={} if global_pts is None else dict(gconf)
    # 旧モデルの最大問題：global lateral_sourcil_Rが反対側へ飛ぶ。
    # 全骨盤のsourcil座標は最終角度に絶対使わず、必ず局所モデル由来に限定。
    for side in ('R','L'):
        for k in ('lateral_sourcil','medial_sourcil'):
            all_points.pop(f'{k}_{side}',None)
            all_confs.pop(f'{k}_{side}',None)
    for side in ('R','L'):
        head=all_points.get(f'femoral_head_{side}')
        result[f'{side}_local_detected']=0
        result[f'{side}_qc']=''
        result[f'{side}_model_center_x']=head[0] if head else np.nan
        if head is None:
            result[f'{side}_qc']='global_head_missing';continue
        crop,x0,y0=crop_square_with_padding(im,head[0],head[1],CROP_SIZE)
        if side=='R':crop=cv2.flip(crop,1)
        local,lconf,lbox=best_kps(local_model,crop,LOCAL_IMGSZ,n=LOCAL_KP)
        if local is None:
            result[f'{side}_qc']='local_hip_missing';continue
        result[f'{side}_local_detected']=1
        for k,p in local.items():
            pp=from_local_point(p,x0,y0,CROP_SIZE,side)
            # 最終角度には局所Poseの推論を採用（全骨盤のlateral_xは使わない）
            dest=f'{k}_{side}' if k in ('femoral_head','lateral_sourcil','medial_sourcil') else f'scar_{side}_{k.split("_")[1]}'
            all_points[dest]=pp
            all_confs[dest]=lconf.get(k,np.nan)
        # 同じ画面側にあるか、左右/遠距離をチェック
        center=all_points.get(f'femoral_head_{side}')
        outer=all_points.get(f'lateral_sourcil_{side}')
        inner=all_points.get(f'medial_sourcil_{side}')
        flags=[]
        if center is None:flags.append('head_missing')
        if outer is None:flags.append('lateral_missing')
        if center is not None and outer is not None:
            dist=np.linalg.norm(np.array(outer[:2])-np.array(center[:2]))
            lateral_dist=(outer[0]-center[0])*(1 if side=='L' else -1)
            if not (12<dist<240):flags.append('head_edge_distance')
            if not (0<lateral_dist<220):flags.append('lateral_order')
        if inner is not None and outer is not None:
            order=(outer[0]-inner[0])*(1 if side=='L' else -1)
            if order < 8:flags.append('roof_point_order')
        result[f'{side}_qc']='|'.join(flags)
        # confidence fields for model reliability
        for k in ('femoral_head','lateral_sourcil','medial_sourcil','scar_medial','scar_lateral'):
            kn=f'{k}_{side}' if not k.startswith('scar_') else f'scar_{side}_{k.split("_")[1]}'
            result[f'{side}_{k}_confidence']=all_confs.get(kn,np.nan)
    # 臨床角度計算は無効な股関節からは算出しない
    for side in ('R','L'):
        for k in ('lcea_deg','fear_like_deg'):
            result[f'{side}_{k}']=np.nan
        if result[f'{side}_qc']=='':
            result[f'{side}_lcea_deg']=signed_lcea(all_points,side)
            result[f'{side}_fear_like_deg']=fear_like(all_points,side)
            if not np.isfinite(result[f'{side}_lcea_deg']):
                result[f'{side}_qc']='LCEA_geometry_or_pelvis_axis'
        result[f'{side}_lcea_qc_pass']=int(np.isfinite(result[f'{side}_lcea_deg']))
        result[f'{side}_fear_qc_warning']=int(np.isfinite(result[f'{side}_fear_like_deg']) and abs(result[f'{side}_fear_like_deg'])>35)
    return result,all_points

G_MODEL=YOLO(str(GLOBAL_PT));L_MODEL=YOLO(str(LOCAL_PT))
HOLDOUT=[r for r in sorted(records,key=lambda q:(q['patient'],q['frame'])) if r['view']=='PA' and r['patient'] in TEST_PATIENTS]
AI_rows=[];AI_points={}
for i,r in enumerate(HOLDOUT):
    row,pts=predict_two_stage(G_MODEL,L_MODEL,r)
    AI_rows.append(row);AI_points[(r['patient'],r['frame'])]=pts
    if (i+1)%10==0:print('AI hold-out inference',i+1,'/',len(HOLDOUT))
AI=pd.DataFrame(AI_rows)
AI.to_csv(ROOT/'AI_holdout_angles_per_frame.csv',index=False)
print('test frames:',len(HOLDOUT))
display(AI.groupby('group').agg(frames=('frame','count'),global_success=('global_detected','sum'),
                                right_lcea_n=('R_lcea_deg','count'),left_lcea_n=('L_lcea_deg','count'),
                                right_fear_n=('R_fear_like_deg','count'),left_fear_n=('L_fear_like_deg','count')))

## 10. AI角度の精度・欠測率を比較（患者単位hold-out）
`LCEA independent annotation`は別カテゴリの2点から得た値。`FEAR-like GT`はsourcil/scarの同一アノテーションを利用します。**欠測はMAEから落ちるため、MAEとともに有効率を必ず表示します。** 画像フレームのMAEだけで患者数を水増ししません。

In [ ]:
angles_long=[];err_kp=[]
for r in HOLDOUT:
    p=r['patient'];f=r['frame']; ai=AI[(AI.patient==p)&(AI.frame==f)].iloc[0]
    predpts=AI_points[(p,f)]
    for side in ('R','L'):
        manual_gt=GT[(GT.patient==p)&(GT.frame==f)&(GT.side==side)].iloc[0]
        for typ,predcol,gtcol in [('LCEA',f'{side}_lcea_deg','lcea_independent_annotation_deg'),
                                 ('FEAR_like',f'{side}_fear_like_deg','fear_like_deg')]:
            pred=float(ai[predcol]) if pd.notna(ai[predcol]) else np.nan
            gt=float(manual_gt[gtcol]) if pd.notna(manual_gt[gtcol]) else np.nan
            angles_long.append({'patient':p,'group':r['group'],'frame':f,'side':side,'metric':typ,
                'manual_deg':gt,'AI_deg':pred,'abs_error_deg':abs(pred-gt) if np.isfinite(pred) and np.isfinite(gt) else np.nan,
                'manual_available':int(np.isfinite(gt)),'AI_available':int(np.isfinite(pred)),
                'qc_reason':ai[f'{side}_qc']})
        for name in ('femoral_head','lateral_sourcil','medial_sourcil','scar_medial','scar_lateral'):
            fullname=f'{name}_{side}' if not name.startswith('scar_') else f'scar_{side}_{name.split("_")[1]}'
            g=getxy(r['points'],fullname);s=getxy(predpts,fullname)
            err_kp.append({'patient':p,'frame':f,'group':r['group'],'side':side,'name':name,
                 'GT_available':int(g is not None),'AI_available':int(s is not None),
                 'error_px':np.linalg.norm(g-s) if g is not None and s is not None else np.nan})
ANGLES=pd.DataFrame(angles_long);ANGLES.to_csv(ROOT/'AI_holdout_vs_manual_angles.csv',index=False)
KP_ERRORS=pd.DataFrame(err_kp);KP_ERRORS.to_csv(ROOT/'AI_holdout_keypoint_errors.csv',index=False)

angle_summary=ANGLES.groupby(['metric','group','side'],as_index=False).agg(
    manual_n=('manual_available','sum'),ai_n=('AI_available','sum'),
    compared_n=('abs_error_deg','count'),MAE_deg=('abs_error_deg','mean'),
    median_abs_error_deg=('abs_error_deg','median'))
angle_summary['valid_fraction_of_GT']=angle_summary['compared_n']/angle_summary['manual_n'].replace(0,np.nan)
angle_summary.to_csv(ROOT/'AI_holdout_angle_summary.csv',index=False)
display(angle_summary.round(3))
display(KP_ERRORS.groupby(['side','name']).error_px.agg(['count','mean','median']).round(2))

# 右lateral sourcilが大幅に反対側へ飛んだ以前の問題を特別に確認
faults=KP_ERRORS[(KP_ERRORS.name=='lateral_sourcil') & (KP_ERRORS.error_px>200)]
print('Lateral sourcil >200px errors:',len(faults))
if len(faults):display(faults.head(30))

## 11. 健常・両側DDHの探索的な患者別グラフ
全患者の**手動参照角度**と、未知患者2名の**AI予測角度**を別レイヤーで描画します。「健常10名」と言っても新LCEA annotationがある健常者は4名のみです。フレームを独立標本として統計検定しません。

In [ ]:
# Group-levelの記述は患者別に左右2股の値を出す。Nを脚数では増やさない。
figdir=ROOT/'figures';figdir.mkdir(exist_ok=True)
for name,col,ylab in [('LCEA','LCEA_median','LCEA (deg)'),('FEAR_like','FEAR_like_median','FEAR-like angle (deg)')]:
    d=GT_SUM[GT_SUM[col].notna()].sort_values(['group','patient','side'])
    fig,ax=plt.subplots(figsize=(10,5))
    for j,(_,z) in enumerate(d.iterrows()):
        group=z['group'];m='o' if group=='Healthy' else 's'
        ax.scatter(j,z[col],marker=m,s=65)
        ax.text(j+.08,z[col],z['patient']+'-'+z['side'],fontsize=8)
    ax.axhline(0,c='grey',ls='--',lw=.8)
    ax.set_xticks(range(len(d)),[z['patient']+'-'+z['side'] for _,z in d.iterrows()],rotation=60)
    ax.set_ylabel(ylab);ax.set_title('Manual annotations only / hip-level medians / no significance tests')
    fig.tight_layout();fig.savefig(figdir/f'manual_{name}_patient_hip.png',dpi=180);plt.show()

for metric in ('LCEA','FEAR_like'):
    f=ANGLES[ANGLES.metric==metric]
    for (patient,side),q in f.groupby(['patient','side']):
        fig,ax=plt.subplots(figsize=(8,4))
        q=q.sort_values('frame')
        ax.plot(q.frame,q.manual_deg,'o-',label='Manual',alpha=.75)
        ax.plot(q.frame,q.AI_deg,'s-',label='AI end-to-end',alpha=.75)
        ax.set_xlabel('DDR frame index (not normalized gait %)')
        ax.set_ylabel(metric+' (deg)');ax.set_title(patient+' '+side+' / holdout')
        ax.legend();fig.tight_layout()
        fig.savefig(figdir/f'{metric}_{patient}_{side}_holdout.png',dpi=160);plt.show()

## 12. AI LCEA/FEAR-like overlay mp4と全フレームCSV
赤：骨頭中心→外側縁（LCEA）、水色：左右teardrop、黄：physeal scarとsourcil線。QCで無効なら角度を**N/A**表示。描画線が外側縁を結んでいるか必ず目視確認してください。

In [ ]:
VIDEO_DIR=ROOT/'video_outputs';VIDEO_DIR.mkdir(exist_ok=True)

def draw_qc_overlay(frame,pts,row):
    out=frame.copy()
    def point(k):
        p=getxy(pts,k)
        return tuple(map(int,np.round(p))) if p is not None else None
    t1,t2=point('teardrop_R'),point('teardrop_L')
    if t1 and t2:cv2.line(out,t1,t2,(255,220,0),2,cv2.LINE_AA)
    for side,color,y in [('R',(0,0,255),42),('L',(0,160,255),80)]:
        head,edge=point(f'femoral_head_{side}'),point(f'lateral_sourcil_{side}')
        if head:cv2.circle(out,head,4,color,-1)
        if edge:cv2.circle(out,edge,4,color,-1)
        if head and edge and row[f'{side}_qc']=='':cv2.line(out,head,edge,color,3,cv2.LINE_AA)
        for names,c in [((f'medial_sourcil_{side}',f'lateral_sourcil_{side}'),(0,255,255)),
                        ((f'scar_{side}_medial',f'scar_{side}_lateral'),(255,240,100))]:
            a,b=point(names[0]),point(names[1])
            if a and b:cv2.line(out,a,b,c,2,cv2.LINE_AA)
        L=row[f'{side}_lcea_deg'];F=row[f'{side}_fear_like_deg'];reason=row[f'{side}_qc']
        txt=f'{side}: LCEA={L:.1f}' if np.isfinite(L) else f'{side}: LCEA=N/A'
        txt+=f' FEAR-like={F:.1f}' if np.isfinite(F) else ' FEAR-like=N/A'
        cv2.putText(out,txt,(15,y),cv2.FONT_HERSHEY_SIMPLEX,.53,color,2,cv2.LINE_AA)
        if reason:cv2.putText(out,reason[:55],(15,y+17),cv2.FONT_HERSHEY_SIMPLEX,.37,color,1,cv2.LINE_AA)
    return out

def save_patient_video(patient):
    rr=[r for r in HOLDOUT if r['patient']==patient]
    rr=sorted(rr,key=lambda r:r['frame'])
    if not rr:return
    f0=cv2.imread(rr[0]['path']);h,w=f0.shape[:2]
    dest=VIDEO_DIR/f'{patient}_two_stage_LCEA_FEAR_like.mp4'
    writer=cv2.VideoWriter(str(dest),cv2.VideoWriter_fourcc(*'mp4v'),15,(w,h))
    if not writer.isOpened():raise RuntimeError('mp4 writer unavailable: '+str(dest))
    for r in rr:
        p=AI_points[(patient,r['frame'])]
        row=AI[(AI.patient==patient)&(AI.frame==r['frame'])].iloc[0]
        writer.write(draw_qc_overlay(cv2.imread(r['path']),p,row))
    writer.release()
    print('Video saved:',dest)
for patient in TEST_PATIENTS:save_patient_video(patient)

# 原稿やスライドに貼れる代表フレームQC
for patient in TEST_PATIENTS:
    rr=[r for r in HOLDOUT if r['patient']==patient]
    if not rr:continue
    r=rr[len(rr)//2];row=AI[(AI.patient==patient)&(AI.frame==r['frame'])].iloc[0]
    image=draw_qc_overlay(cv2.imread(r['path']),AI_points[(patient,r['frame'])],row)
    cv2.imwrite(str(figdir/f'AI_overlay_{patient}_QC.png'),image)

## 13. 【任意】新規7例の患者単位LOSO-CV
7患者のうち1患者を毎回テスト、残りから1患者を検証として選び、全骨盤＋局所モデルを**そのfold専用に新規学習**します。**holdoutの既存学習済みモデルを流用しません**（テスト患者の学習情報が漏れるため）。計算時間が長いので学会2週間前はまず固定holdoutを完成させ、余裕があれば有効化します。

In [ ]:
LOSO_DIR=ROOT/'LOSO_new7';LOSO_DIR.mkdir(exist_ok=True)

def choose_val_for_fold(test):
    # DDHテスト→健常val、健常テスト→DDHvalに固定しやすいが、testと重複しないこと
    choices=['DDH002','DDH001','009','008','010','006','DDH003']
    return next(x for x in choices if x!=test)

lovo_summary=[]
if RUN_LOSO:
    # holdoutの2モデルを破棄してLOSOの学習VRAMを確保する
    del G_MODEL,L_MODEL
    cleanup_gpu('before LOSO')
    tests=list(LOSO_CASES)
    if LOSO_MAX_FOLDS is not None:tests=tests[:int(LOSO_MAX_FOLDS)]
    for i,test in enumerate(tests):
        val=choose_val_for_fold(test)
        print('\n=== LOSO',i+1,'test',test,'val',val,'===')
        fold=LOSO_DIR/f'test_{test}'
        gdata,gyml,_=prepare_global('PA',test=(test,),val=(val,),root=fold/'global_data')
        ldata,lyml,_,_=prepare_local(test=(test,),val=(val,),root=fold/'local_data',aug=True)
        gp=train_pose(gyml,fold/'runs'/'global',LOSO_GLOBAL_EPOCHS,GLOBAL_IMGSZ,GLOBAL_BATCH,
                      True,False,seed=SEED+i)
        lp=train_pose(lyml,fold/'runs'/'local',LOSO_LOCAL_EPOCHS,LOCAL_IMGSZ,LOCAL_BATCH,
                      True,False,seed=SEED+i)
        gm=YOLO(str(gp));lm=YOLO(str(lp))
        foldrows=[]
        for r in records:
            if r['view']!='PA' or r['patient']!=test:continue
            out,_=predict_two_stage(gm,lm,r)
            for side in ('R','L'):
                man=GT[(GT.patient==test)&(GT.frame==r['frame'])&(GT.side==side)].iloc[0]
                for name,pre,ref in [('LCEA',out[f'{side}_lcea_deg'],man.lcea_independent_annotation_deg),
                                      ('FEAR_like',out[f'{side}_fear_like_deg'],man.fear_like_deg)]:
                    foldrows.append({'test_patient':test,'group':r['group'],'frame':r['frame'],
                        'side':side,'metric':name,'pred':pre,'manual':ref,
                        'abs_error':abs(pre-ref) if np.isfinite(pre) and np.isfinite(ref) else np.nan})
        fd=pd.DataFrame(foldrows);fd.to_csv(fold/'LOSO_fold_angles.csv',index=False)
        lovo_summary.append(fd)
        del gm,lm;cleanup_gpu('after LOSO fold')
    if lovo_summary:
        ld=pd.concat(lovo_summary,ignore_index=True)
        ld.to_csv(LOSO_DIR/'LOSO_all_folds_angles.csv',index=False)
        summary=ld.groupby(['test_patient','group','metric'],as_index=False).agg(
            test_frames=('frame','nunique'),reference_angles=('manual','count'),
            n_compared=('abs_error','count'),MAE_deg=('abs_error','mean'))
        summary.to_csv(LOSO_DIR/'LOSO_patient_level_summary.csv',index=False)
        display(summary)
else:
    print('RUN_LOSO=False: 患者単位LOSOは実行していません。実施したと発表しないでください。')

## 14. 【任意】側面ML（別研究の9点）を今回のZIPだけで新規学習
LCEA/FEARはPA正面画像の指標です。MLは独立の動態計測モデルとしてのみ扱い、**PAの2D角度をそのまま3D角度に変換しません**。学会優先ならデフォルトOFF。

In [ ]:
if TRAIN_ML:
    ML_DATA,ML_YAML,ml_valid=prepare_global('ML')
    print('ML labels:',ml_valid)
    ML_PT=train_pose(ML_YAML,ROOT/'runs'/'ML_9kp',ML_EPOCHS,GLOBAL_IMGSZ,GLOBAL_BATCH,
                     True,FORCE_RETRAIN)
    ml_model=YOLO(str(ML_PT))
    m=ml_model.val(data=str(ML_YAML),split='test',imgsz=GLOBAL_IMGSZ,batch=1,
                   device=0 if torch.cuda.is_available() else 'cpu',plots=False)
    pd.DataFrame([{'pose_mAP50':float(m.pose.map50),'pose_mAP50_95':float(m.pose.map)}]).to_csv(ROOT/'ML_holdout_pose_metrics.csv',index=False)
    del ml_model,m;cleanup_gpu('after ML')
else:
    print('TRAIN_ML=False: MLモデルは今回学習していません。')

## 15. 学会用の最終チェックと成果物ZIP
**最重要**：左右LCEAに非現実的な負値が出ていないか、`lateral_sourcil_R`位置誤差、QC除外率、AI対手動のMAE、左右のoverlayを確認してから発表してください。**自動計測が成立しなかった場合は成功と書かず、手動アノテーションによる探索結果とAI開発結果を区別してください。**

In [ ]:
qc_summary={'total_PA_frames':int((manifest.view=='PA').sum()),
            'total_ML_frames':int((manifest.view=='ML').sum()),
            'all_unique_patients':int(manifest.patient.nunique()),
            'holdout_patients':','.join(TEST_PATIENTS),
            'holdout_PA_frames':len(HOLDOUT),
            'AI_model_type':'10kp global + 5kp mirrored local (no prior hip weights)',
            'LOSO_executed':bool(RUN_LOSO), 'ML_training_executed':bool(TRAIN_ML),
            'right_lcea_valid_n':int(AI.R_lcea_deg.notna().sum()),
            'left_lcea_valid_n':int(AI.L_lcea_deg.notna().sum()),
            'right_fear_valid_n':int(AI.R_fear_like_deg.notna().sum()),
            'left_fear_valid_n':int(AI.L_fear_like_deg.notna().sum()),
            'total_right_qc_fail':int(AI.R_qc.ne('').sum()),
            'total_left_qc_fail':int(AI.L_qc.ne('').sum())}
qcs=pd.DataFrame([qc_summary]);qcs.to_csv(ROOT/'CONFERENCE_QC_SUMMARY.csv',index=False)
display(qcs.T)

# 大容量のYOLO画像・学習中間データを除き、重要なCSV / Fig / weights / mp4 をまとめる
archive=Path('/content/DDR_DDH_Revised_Conference_Outputs.zip')
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=5) as z:
    for file in ROOT.rglob('*'):
        if not file.is_file():continue
        if any(x in file.parts for x in ('yolo_data','raw')):continue
        if file.suffix.lower() not in ('.csv','.png','.jpg','.mp4','.yaml','.txt','.pt'):continue
        if file.name=='last.pt':continue
        # 厳密CV時は必要なptが多数になるため、best.ptだけ含める
        z.write(file,arcname=str(file.relative_to(ROOT)))
print('Results ZIP:',archive,round(archive.stat().st_size/1024**2,2),'MB')

try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    print('Colabでは files.download で成果物ZIPを取得してください')

## 研究の報告原則（学会発表用）

* 「健常10例、両側DDH3例、PA234フレーム、ML254フレーム」はデータ収集として記述可能。ただし健常PAは9例のみ、LCEA/FEARの全手動参照は健常4例＋DDH3例。
* 初期hold-outは健常010とDDH003（合計38フレーム）。AI精度の表では**患者数N=2**を必ず併記。
* ローカルPoseのROI位置はAIが検出した骨頭中心を使用し、評価時にGT中心を利用しない。
* LCEAはDDRの**動的投影角度**、FEARは原著符号未照合の**FEAR-like指標**。臨床的診断カットオフの主張はしない。
* 健常とDDHの比較は探索的な患者ごとの角度分布・左右トレースに留め、有意差検定や診断精度のAUCは計算しない。
* 手動由来の群間差と、AIの自動測定成功率・MAEを混同しない。